# GBIF practical: clean data, map occurrences, and build a simple SDM

This notebook combines the workflow from the three R scripts with the guidance from `README.md`.

In this practical, you will:

- clean GBIF occurrence records
- compare a simple filtering workflow with `CoordinateCleaner`
- map your cleaned records in Austria
- fit a simple climate-based species distribution model (SDM)

This is a teaching workflow. It helps you understand each step, but it is not a publication-ready analysis.

## Before you start

Place your GBIF download in `data/`. If you do not want to download a species yourself right now, you can use one of the example files there. 
You will mainly work with these folders:

- `data/` for input files and downloaded spatial data
- `results/` for cleaned tables, maps, rasters, and the model summary
- `backup/` for backup data, just in case you get lost somewhere.

## 1. Install packages if you need them

Run the next cell only if some packages are missing in your R environment.

**Here this is NOT NECESSARY because all the packages are already installed**

In [ ]:
# install.packages(c("dplyr", "ggplot2", "terra", "geodata", "sf"))
# install.packages("CoordinateCleaner")

## 2. Load packages and define your settings

Here we will set some basic parameters for our model.

Edit the file name and species name before you continue.

In [12]:
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(terra)
  library(geodata)
  library(sf)
})

# Place your input files in data/ using drag and drop and modify the names below:
input_file <- "data/Pinus_cembra_AT.csv"
species_name <- "Pinus cembra"

# some parameters for the analysis
max_uncertainty_m <- 10000 # allowed coordinate uncertainty of the record in m
min_year <- 1970 # age of the record
nrecords <- 5000 # number of records to be used

n_background <- 5000 # number of background points
climate_resolution <- 10
point_colour <- "#1b9e77"
point_size <- 1.6

dir.create("results", recursive = TRUE, showWarnings = FALSE)

## 3. Read the GBIF file

Now we will load the GBIF occurence data file into memory.

Important: GBIF Simple CSV downloads are usually tab delimited. If your file uses commas instead, change `sep = "\t"` to `sep = ","`.

In [3]:
gbif_raw <- read.delim(
  input_file,
  sep = "\t",
  quote = "",
  fill = TRUE,
  stringsAsFactors = FALSE
)

gbif_raw <- gbif_raw %>%
  mutate(
    decimalLongitude = suppressWarnings(as.numeric(decimalLongitude)),
    decimalLatitude = suppressWarnings(as.numeric(decimalLatitude)),
    coordinateUncertaintyInMeters = suppressWarnings(as.numeric(coordinateUncertaintyInMeters)),
    year = suppressWarnings(as.integer(year))
  )

cat("Rows in original file:", nrow(gbif_raw), "")
if (nrow(gbif_raw) > 5000) {
    gbif <- gbif_raw[sample(nrow(gbif_raw), nrecords), ]
    } else { gbif <- gbif_raw }
cat("Rows after sampling file:", nrow(gbif), "")
head(gbif)

Rows in original file: 3153 Rows after sampling file: 3153 

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,⋯,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>
1,6353256753,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/365977773,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Martin,2026-05-28T08:15:12,CC_BY_NC_4_0,Martin,Martin,NA,,2026-06-06T03:59:58.092Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
2,6353175628,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/366335780,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Tina,2026-05-29T14:05:50,CC0_1_0,Tina,Tina,NA,,2026-06-06T04:00:02.783Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
3,6352956317,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/344159000,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Martin A. Prinz,2026-05-27T08:43:14,CC_BY_NC_4_0,Soňa Kovalovská,Soňa Kovalovská,NA,,2026-06-06T03:53:02.946Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
4,6346596639,7b904404-f762-11e1-a439-00145eb45e9a,56C8C4E5-F332-407D-AE81-9E283849AE34,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Schagowetz Max,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Schagowetz Max,NA,,2026-06-03T12:48:50.069Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
5,6346590694,7b904404-f762-11e1-a439-00145eb45e9a,8AF2F514-4C7F-4D2F-AF19-78AB92BDA4A3,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Sarnthein Rudolf,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Sarnthein Rudolf,NA,,2026-06-03T12:48:45.597Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
6,6346579196,7b904404-f762-11e1-a439-00145eb45e9a,8A601349-6880-42F7-A3E2-EB0F769CE6DB,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Morass Peter,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Morass Peter,NA,,2026-06-03T12:48:33.323Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND


## 4. Apply some basic filters

Records on GBIF can have different problems. Therefore, we will first have to filter them somehow to exclude potentially problematic entries.

You will:

- keep only your focal species (in case you downloaded data from multiple species)
- keep only Austrian records
- require usable coordinates
- remove records with very large coordinate uncertainty
- keep common occurrence record types
- remove very old records

In [13]:
basic_dat <- gbif %>%
  filter(species == species_name) %>%
  filter(countryCode == "AT") %>%
  filter(!is.na(decimalLongitude), !is.na(decimalLatitude)) %>%
  filter(between(decimalLongitude, -180, 180), between(decimalLatitude, -90, 90)) %>%
  filter(is.na(coordinateUncertaintyInMeters) | coordinateUncertaintyInMeters <= max_uncertainty_m) %>%
  filter(is.na(basisOfRecord) | basisOfRecord %in% c("HUMAN_OBSERVATION", "OBSERVATION", "PRESERVED_SPECIMEN")) %>%
  filter(is.na(year) | year >= min_year)

cat("Rows after shared basic filters:", nrow(basic_dat), "
")

# if this happens your filtering was too stringent
if (nrow(basic_dat) == 0) {
    basic_dat = gbif
  stop("No records left after the basic filters. Check your file name, species name, and filters.")

}

head(basic_dat)

Rows after shared basic filters: 2799 


,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,⋯,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>
1,6353256753,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/365977773,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Martin,2026-05-28T08:15:12,CC_BY_NC_4_0,Martin,Martin,NA,,2026-06-06T03:59:58.092Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
2,6353175628,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/366335780,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Tina,2026-05-29T14:05:50,CC0_1_0,Tina,Tina,NA,,2026-06-06T04:00:02.783Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
3,6352956317,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/344159000,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Martin A. Prinz,2026-05-27T08:43:14,CC_BY_NC_4_0,Soňa Kovalovská,Soňa Kovalovská,NA,,2026-06-06T03:53:02.946Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
4,6346596639,7b904404-f762-11e1-a439-00145eb45e9a,56C8C4E5-F332-407D-AE81-9E283849AE34,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Schagowetz Max,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Schagowetz Max,NA,,2026-06-03T12:48:50.069Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
5,6346579196,7b904404-f762-11e1-a439-00145eb45e9a,8A601349-6880-42F7-A3E2-EB0F769CE6DB,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Morass Peter,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Morass Peter,NA,,2026-06-03T12:48:33.323Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
6,6346564864,7b904404-f762-11e1-a439-00145eb45e9a,14726ECB-98E7-42FA-A1AA-445AA5440ADA,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Moll B.,NA,,2026-06-03T12:48:32.851Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND


## 5. A little bit stricter filtering

We are not yet done with filtering. One almost never is...

Lets make it a bit more strict.

What will happen now:

- keep only points inside a simple Austria bounding box
- remove duplicated coordinate pairs for the same species

In [14]:
standard_clean_dat <- basic_dat %>%
  filter(between(decimalLongitude, 9, 18), between(decimalLatitude, 46, 50)) %>%
  distinct(species, decimalLongitude, decimalLatitude, .keep_all = TRUE)

write.csv(standard_clean_dat, "results/gbif_clean_occurrences_standard.csv", row.names = FALSE)
cat("Rows after standard cleaning:", nrow(standard_clean_dat), "
")
head(standard_clean_dat)

Rows after standard cleaning: 2092 


,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,⋯,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>
1,6353256753,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/365977773,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Martin,2026-05-28T08:15:12,CC_BY_NC_4_0,Martin,Martin,NA,,2026-06-06T03:59:58.092Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
2,6353175628,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/366335780,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Tina,2026-05-29T14:05:50,CC0_1_0,Tina,Tina,NA,,2026-06-06T04:00:02.783Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
3,6352956317,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/344159000,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Martin A. Prinz,2026-05-27T08:43:14,CC_BY_NC_4_0,Soňa Kovalovská,Soňa Kovalovská,NA,,2026-06-06T03:53:02.946Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
4,6346596639,7b904404-f762-11e1-a439-00145eb45e9a,56C8C4E5-F332-407D-AE81-9E283849AE34,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Schagowetz Max,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Schagowetz Max,NA,,2026-06-03T12:48:50.069Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
5,6346579196,7b904404-f762-11e1-a439-00145eb45e9a,8A601349-6880-42F7-A3E2-EB0F769CE6DB,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,Morass Peter,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Morass Peter,NA,,2026-06-03T12:48:33.323Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND
6,6346564864,7b904404-f762-11e1-a439-00145eb45e9a,14726ECB-98E7-42FA-A1AA-445AA5440ADA,Plantae,Tracheophyta,Pinopsida,Pinales,Pinaceae,Pinus,Pinus cembra,⋯,,,CC_BY_4_0,Tiroler Landesmuseen-Betriebsgesellschaft m.b.H.,Moll B.,NA,,2026-06-03T12:48:32.851Z,,COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NOT_FOUND


## 6. Optional: clean records with CoordinateCleaner

`CoordinateCleaner` flags common spatial problems such as:

- country centroids
- capitals
- biodiversity institutions
- GBIF headquarters
- sea points
- equal latitude and longitude
- duplicate coordinates
- spatial outliers
- zero coordinates

If you do not have the package installed, the cell will skip this step.

In [ ]:
suppressPackageStartupMessages({
  library(CoordinateCleaner)
})

In [ ]:
coordinatecleaner_clean_dat <- NULL
coordinatecleaner_flags <- NULL

if (requireNamespace("CoordinateCleaner", quietly = TRUE)) {
  cc_result <- CoordinateCleaner::clean_coordinates(
    x = basic_dat,
    lon = "decimalLongitude",
    lat = "decimalLatitude",
    species = "species",
    tests = c(
      "capitals",
      "centroids",
      "duplicates",
      "equal",
      "gbif",
      "institutions",
      "outliers",
      "seas",
      "zeros"
    ),
    value = "spatialvalid",
    verbose = TRUE
  )

  coordinatecleaner_flags <- as.data.frame(cc_result)
  coordinatecleaner_clean_dat <- coordinatecleaner_flags %>%
    filter(.summary) %>%
    select(-starts_with("."))

  write.csv(coordinatecleaner_clean_dat, "results/gbif_clean_occurrences_coordinatecleaner.csv", row.names = FALSE)
  write.csv(coordinatecleaner_flags, "results/gbif_coordinatecleaner_flags.csv", row.names = FALSE)

  cat("Rows after CoordinateCleaner:", nrow(coordinatecleaner_clean_dat), "
")
  head(coordinatecleaner_clean_dat)
} else {
  message("CoordinateCleaner is not installed, so this comparison step was skipped.")
}

## 7. Choose the cleaned dataset you want to use

Set `active_cleaning_method` here.

- Use `"standard"` if you want the simple filtering result.
- Use `"coordinatecleaner"` if you want the automated cleaning result.

In [ ]:
active_cleaning_method <- "standard"   # use "standard" or "coordinatecleaner"

if (active_cleaning_method == "standard") {
  clean_dat <- standard_clean_dat
} else if (active_cleaning_method == "coordinatecleaner") {
  if (is.null(coordinatecleaner_clean_dat)) {
    stop("You selected 'coordinatecleaner', but CoordinateCleaner is not available.")
  }
  clean_dat <- coordinatecleaner_clean_dat
} else {
  stop("active_cleaning_method must be either 'standard' or 'coordinatecleaner'.")
}

if (nrow(clean_dat) == 0) {
  stop("No records left after the selected cleaning approach.")
}

write.csv(clean_dat, "results/gbif_clean_occurrences.csv", row.names = FALSE)
cat("Selected cleaning method:", active_cleaning_method, "
")
cat("Rows in selected cleaned dataset:", nrow(clean_dat), "
")

## 8. Plot the cleaned occurrence records

A map is not just a figure. Use it to check whether your points are really in Austria and whether you still see suspicious clusters or outliers.

In [ ]:
occ <- read.csv("results/gbif_clean_occurrences.csv", stringsAsFactors = FALSE)

austria <- geodata::gadm(country = "AUT", level = 0, path = "data")
austria_sf <- sf::st_as_sf(austria)

species_label <- unique(occ$species)
if (length(species_label) > 1) species_label <- paste(length(species_label), "species")
if (length(species_label) == 0) species_label <- "selected species"

p_occ <- ggplot() +
  geom_sf(data = austria_sf, fill = "grey95", colour = "grey35") +
  geom_point(
    data = occ,
    aes(x = decimalLongitude, y = decimalLatitude),
    colour = point_colour,
    size = point_size,
    alpha = 0.75
  ) +
  coord_sf(xlim = c(9, 18), ylim = c(46, 50), expand = FALSE) +
  labs(
    title = paste("GBIF occurrences in Austria:", species_label),
    subtitle = paste("Clean records:", nrow(occ)),
    x = "Longitude",
    y = "Latitude"
  ) +
  theme_minimal(base_size = 12)

print(p_occ)
ggsave("results/occurrence_map_austria.png", p_occ, width = 6, height = 7, dpi = 300)

## 9. Prepare the data for the SDM

This model uses:

- cleaned GBIF occurrences as presences
- random background points within Austria
- two WorldClim predictors: annual mean temperature (`bio1`) and annual precipitation (`bio12`)
- a simple logistic regression with `glm`

The background points are not true absences. They are just a contrast sample from the available environment.

In [ ]:
occ_sdm <- read.csv("results/gbif_clean_occurrences.csv", stringsAsFactors = FALSE) %>%
  filter(species == species_name)
occ_sdm <- occ
cat("Clean records available for SDM:", nrow(occ_sdm), "
")

if (nrow(occ_sdm) < 20) {
  stop("This simple SDM needs at least 20 cleaned records.")
}

austria <- geodata::gadm(country = "AUT", level = 0, path = "data")
austria_vect <- austria
austria_sf <- sf::st_as_sf(austria)

clim <- geodata::worldclim_global(var = "bio", res = climate_resolution, path = "data")
clim_at <- terra::mask(terra::crop(clim, austria_vect), austria_vect)
preds <- clim_at[[grep("bio_1$|bio_12$", names(clim_at))]]
names(preds) <- c("bio1", "bio12")

preds

## 10. Extract climate data for presences and sample background points

You now link occurrence coordinates to climate values and then sample random background points inside Austria.

In [ ]:
pres_xy <- occ_sdm[, c("decimalLongitude", "decimalLatitude")]
pres_env <- terra::extract(preds, pres_xy, ID = FALSE)
pres_dat <- cbind(pa = 1, pres_env) %>% na.omit()

bg_points <- sf::st_sample(austria_sf, size = n_background, type = "random")
bg_xy <- as.data.frame(sf::st_coordinates(bg_points))
names(bg_xy) <- c("decimalLongitude", "decimalLatitude")
bg_env <- terra::extract(preds, bg_xy, ID = FALSE)
bg_dat <- cbind(pa = 0, bg_env) %>% na.omit()

cat("Presence rows used in the model:", nrow(pres_dat), "
")
cat("Background rows used in the model:", nrow(bg_dat), "
")
head(pres_dat)

## 11. Fit the simple SDM

The model is deliberately simple so you can inspect every part of it.

In [ ]:
model_dat <- bind_rows(pres_dat, bg_dat)

sdm <- glm(pa ~ bio1 + bio12, data = model_dat, family = binomial())
summary(sdm)

## 12. Predict suitability across Austria and create a binary map

First you predict continuous climatic suitability. Then you apply a simple teaching threshold: the 10th percentile of predicted suitability at the presence points.

In [ ]:
suitability <- terra::predict(preds, sdm, type = "response")
pres_scores <- predict(sdm, newdata = pres_dat, type = "response")
threshold <- as.numeric(quantile(pres_scores, probs = 0.10, na.rm = TRUE))
binary_map <- suitability >= threshold

terra::writeRaster(suitability, "results/sdm_probability_map.tif", overwrite = TRUE)
terra::writeRaster(binary_map, "results/sdm_binary_map.tif", overwrite = TRUE)
writeLines(capture.output(summary(sdm)), "results/sdm_model_summary.txt")

cat("Threshold:", round(threshold, 3), "
")

## 13. Plot the SDM output

Interpret this as modelled climatic suitability, not as a guaranteed distribution map.

In [ ]:
plot_dat <- as.data.frame(suitability, xy = TRUE, na.rm = TRUE)
names(plot_dat)[3] <- "suitability"

p_sdm <- ggplot() +
  geom_raster(data = plot_dat, aes(x = x, y = y, fill = suitability)) +
  geom_sf(data = austria_sf, fill = NA, colour = "grey20", linewidth = 0.3) +
  geom_point(
    data = occ_sdm,
    aes(x = decimalLongitude, y = decimalLatitude),
    colour = "black",
    size = 0.8,
    alpha = 0.6
  ) +
   scale_fill_gradient(name = "Suitability", low="white", high="red", limits=c(0,1)) +
  coord_sf(xlim = c(9, 18), ylim = c(46, 50), expand = FALSE) +
  labs(
    title = paste("Simple SDM for", species_name),
    subtitle = paste("GLM with bio1 and bio12 | threshold =", round(threshold, 3)),
    x = "Longitude",
    y = "Latitude"
  ) +
  theme_minimal(base_size = 12)

print(p_sdm)
ggsave("results/sdm_probability_map.png", p_sdm, width = 6, height = 7, dpi = 300)

## 14. What you should check when you interpret your result

Keep these points in mind:

- Your GBIF records may still reflect sampling bias.
- Your background points are not true absences.
- This model uses only two climate predictors.
- The thresholded binary map is only one possible simplification.
- A useful-looking map is not the same thing as a validated ecological model.

For this practical, focus on understanding the workflow and on checking whether the result is biologically plausible.

## References

- Elith, J. & Leathwick, J. R. 2009. *Annual Review of Ecology, Evolution, and Systematics* 40: 677–697.
- Phillips, S. J. et al. 2009. *Ecological Applications* 19: 181–197.
- Roberts, D. R. et al. 2017. *Ecography* 40: 913–929.
- Fiorentino, D. et al. 2025. *Ecological Modelling* 508: 111207.